In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

df = pd.read_csv("blackjack_uniform_data.csv")
print(df.shape)
display(df.head())

(69006, 9)


,player_sum,dealer_showing,usable_ace,action,reward,next_player_sum,next_dealer_showing,next_usable_ace,done
0,14,3,0,0,-1.0,14,3,0,1
1,10,8,0,0,1.0,10,8,0,1
2,12,3,0,1,0.0,15,3,0,0
3,15,3,0,0,-1.0,15,3,0,1
4,20,10,0,0,1.0,20,10,0,1


In [2]:
def reconstruct_episodes(df):
    episodes = []
    current_episode = []

    for _, row in df.iterrows():
        transition = {
            "state": (
                int(row["player_sum"]),
                int(row["dealer_showing"]),
                int(row["usable_ace"]),
            ),
            "action": int(row["action"]),
            "reward": float(row["reward"]),
            "next_state": (
                int(row["next_player_sum"]),
                int(row["next_dealer_showing"]),
                int(row["next_usable_ace"]),
            ),
            "done": int(row["done"]),
        }

        current_episode.append(transition)

        if transition["done"] == 1:
            episodes.append(current_episode)
            current_episode = []

    if len(current_episode) > 0:
        episodes.append(current_episode)

    return episodes

episodes = reconstruct_episodes(df)
print("Number of reconstructed episodes:", len(episodes))
print("Length of first episode:", len(episodes[0]))
print("First transition of first episode:", episodes[0][0])

Number of reconstructed episodes: 50000
Length of first episode: 1
First transition of first episode: {'state': (14, 3, 0), 'action': 0, 'reward': -1.0, 'next_state': (14, 3, 0), 'done': 1}


In [3]:
theta = np.zeros((32, 11, 2, 2), dtype=np.float64)

def softmax(x):
    z = x - np.max(x)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)

def get_probs(state):
    player, dealer, usable = state
    return softmax(theta[player, dealer, usable])

def greedy_action(state):
    probs = get_probs(state)
    return int(np.argmax(probs))

In [4]:
def compute_returns_from_episode(ep, gamma=1.0):
    rewards = [tr["reward"] for tr in ep]
    G = 0.0
    returns = []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return returns

In [5]:
def offpolicy_reinforce(
    episodes,
    alpha=0.001,
    gamma=1.0,
    n_passes=20,
    clip_rho=10.0
):
    global theta

    avg_weight_history = []
    avg_return_history = []

    for epoch in range(n_passes):
        total_weight = 0.0
        total_final_return = 0.0

        for ep in episodes:
            returns = compute_returns_from_episode(ep, gamma=gamma)

            # trajectory importance ratio
            rho = 1.0
            for tr in ep:
                state = tr["state"]
                action = tr["action"]
                pi_prob = get_probs(state)[action]
                mu_prob = 0.5   # uniform behavior policy
                rho *= pi_prob / mu_prob

            # clip to reduce variance
            rho = min(rho, clip_rho)

            total_weight += rho
            total_final_return += sum(tr["reward"] for tr in ep)

            # gradient ascent
            for tr, G in zip(ep, returns):
                state = tr["state"]
                action = tr["action"]
                probs = get_probs(state)

                player, dealer, usable = state

                for k in range(2):
                    indicator = 1.0 if k == action else 0.0
                    grad_log = indicator - probs[k]
                    theta[player, dealer, usable, k] += alpha * rho * G * grad_log

        avg_weight_history.append(total_weight / len(episodes))
        avg_return_history.append(total_final_return / len(episodes))

        print(
            f"Epoch {epoch+1}/{n_passes}, "
            f"avg clipped rho = {avg_weight_history[-1]:.4f}, "
            f"avg dataset return = {avg_return_history[-1]:.4f}"
        )

    return avg_weight_history, avg_return_history

In [6]:
theta = np.zeros((32, 11, 2, 2), dtype=np.float64)

weight_hist, return_hist = offpolicy_reinforce(
    episodes,
    alpha=0.001,
    gamma=1.0,
    n_passes=30,
    clip_rho=10.0
)

Epoch 1/30, avg clipped rho = 0.9990, avg dataset return = -0.4011
Epoch 2/30, avg clipped rho = 0.9974, avg dataset return = -0.4011
Epoch 3/30, avg clipped rho = 0.9966, avg dataset return = -0.4011
Epoch 4/30, avg clipped rho = 0.9961, avg dataset return = -0.4011
Epoch 5/30, avg clipped rho = 0.9957, avg dataset return = -0.4011
Epoch 6/30, avg clipped rho = 0.9955, avg dataset return = -0.4011
Epoch 7/30, avg clipped rho = 0.9953, avg dataset return = -0.4011
Epoch 8/30, avg clipped rho = 0.9951, avg dataset return = -0.4011
Epoch 9/30, avg clipped rho = 0.9950, avg dataset return = -0.4011
Epoch 10/30, avg clipped rho = 0.9949, avg dataset return = -0.4011
Epoch 11/30, avg clipped rho = 0.9948, avg dataset return = -0.4011
Epoch 12/30, avg clipped rho = 0.9947, avg dataset return = -0.4011
Epoch 13/30, avg clipped rho = 0.9946, avg dataset return = -0.4011
Epoch 14/30, avg clipped rho = 0.9945, avg dataset return = -0.4011
Epoch 15/30, avg clipped rho = 0.9944, avg dataset return

In [9]:
def evaluate_policy(theta, n_eval_episodes=100000, policy_type="greedy"):
    eval_env = gym.make("Blackjack-v1", natural=False, sab=False)

    def get_probs_eval(state):
        player, dealer, usable = state
        usable = int(usable)
        z = theta[player, dealer, usable]
        z = z - np.max(z)
        e = np.exp(z)
        return e / np.sum(e)

    returns = []

    for _ in range(n_eval_episodes):
        state, _ = eval_env.reset()
        done = False
        total_reward = 0.0

        while not done:
            if policy_type == "greedy":
                probs = get_probs_eval(state)
                action = int(np.argmax(probs))

            elif policy_type == "stochastic":
                probs = get_probs_eval(state)
                action = int(np.random.choice([0, 1], p=probs))

            elif policy_type == "uniform":
                action = int(np.random.choice([0, 1]))

            else:
                raise ValueError("policy_type must be 'greedy', 'stochastic', or 'uniform'")

            state, reward, terminated, truncated, _ = eval_env.step(action)
            done = terminated or truncated
            total_reward += reward

        returns.append(total_reward)

    eval_env.close()

    returns = np.array(returns, dtype=np.float64)
    return {
        "avg_return": returns.mean(),
        "win_rate": np.mean(returns > 0),
        "draw_rate": np.mean(returns == 0),
        "loss_rate": np.mean(returns < 0),
        "returns": returns,
    }

In [10]:
results = evaluate_policy(theta, n_eval_episodes=5000, policy_type="greedy")
print("Greedy evaluation")
print(f"Average return: {results['avg_return']:.4f}")
print(f"Win rate:       {results['win_rate']:.4f}")
print(f"Draw rate:      {results['draw_rate']:.4f}")
print(f"Loss rate:      {results['loss_rate']:.4f}")

results = evaluate_policy(theta, n_eval_episodes=5000, policy_type="stochastic")
print("\nStochastic evaluation")
print(f"Average return: {results['avg_return']:.4f}")
print(f"Win rate:       {results['win_rate']:.4f}")
print(f"Draw rate:      {results['draw_rate']:.4f}")
print(f"Loss rate:      {results['loss_rate']:.4f}")

results = evaluate_policy(theta, n_eval_episodes=5000, policy_type="uniform")
print("\nUniform evaluation")
print(f"Average return: {results['avg_return']:.4f}")
print(f"Win rate:       {results['win_rate']:.4f}")
print(f"Draw rate:      {results['draw_rate']:.4f}")
print(f"Loss rate:      {results['loss_rate']:.4f}")

Greedy evaluation
Average return: -0.0696
Win rate:       0.4204
Draw rate:      0.0896
Loss rate:      0.4900

Stochastic evaluation
Average return: -0.1244
Win rate:       0.3994
Draw rate:      0.0768
Loss rate:      0.5238

Uniform evaluation
Average return: -0.3906
Win rate:       0.2828
Draw rate:      0.0438
Loss rate:      0.6734
